# Benchmark RAGAS — Assistant RAG multimodal

Ce notebook évalue la qualité du système RAG du projet avec **RAGAS**, en réutilisant
**exactement le même pipeline que l'API** (`/api/chat`) :

1. embedding de la question (`hf_client.embed_texts`),
2. retrieval hybride dense + BM25 (`vector_store.hybrid_query`),
3. génération de la réponse sourcée (`mistral_client.generate_answer`).

Les métriques calculées :

| Métrique | Ce qu'elle mesure | A besoin de |
|---|---|---|
| **Faithfulness** | La réponse est-elle fidèle aux passages récupérés (pas d'hallucination) ? | réponse + contextes |
| **Response relevancy** | La réponse répond-elle vraiment à la question ? | réponse + embeddings |
| **Context precision** | Les passages récupérés sont-ils pertinents (peu de bruit) ? | contextes + référence |
| **Context recall** | Le retrieval a-t-il ramené l'info nécessaire pour répondre ? | contextes + référence |

**Choix d'implantation** : RAGAS a besoin d'un LLM « juge » et d'un modèle d'embeddings.
On réutilise les modèles **déjà utilisés par le projet** (Mistral comme juge, `multilingual-e5-large`
via l'Inference API HF comme embeddings) plutôt qu'OpenAI : pas de provider supplémentaire,
et le juge parle la même langue technique que le système évalué.

## 0. Dépendances

À exécuter une seule fois. Installation en scope utilisateur (pas de droits admin requis,
cf. contraintes PC verrouillé). `ragas` tire `langchain-core`, `datasets`, etc.

In [ ]:
# %pip install --user ragas langchain-mistralai pandas

     ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
     ----------- ---------------------------- 0.5/1.8 MB 5.7 MB/s eta 0:00:01
     ---------------------------------------- 1.8/1.8 MB 8.3 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: still running...
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached orjson-3.11.9-cp314-cp314-win_amd64.whl.metadata (43 kB)
  Using cached aiohttp-3.14.1-cp314-cp314-win_amd64.whl.metadata (8.5 kB)
  Using cached aiohappyeyeballs-2.6.2-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp314-cp

  error: subprocess-exited-with-error
  
  × Building wheel for scikit-network (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [332 lines of output]
      pip_system_certs: ERROR: truststore not available
      Compiling ./sknetwork\classification\vote.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\classification\vote.pyx
      Compiling ./sknetwork\clustering\leiden_core.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\clustering\leiden_core.pyx
      Compiling ./sknetwork\clustering\louvain_core.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\clustering\louvain_core.pyx
      Compiling ./sknetwork\hierarchy\paris.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\hierarchy\paris.pyx
      Compiling ./sknetwork\linalg\diteration.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\linalg\diteration.pyx
      Compiling ./sknetwork\linalg\push.pyx because it changed.
      [1/1] Cythonizing ./sknetwork\linalg\push.pyx

## 1. Setup : chemins, SSL, et chargement du pipeline du backend

On se place dans `backend/` pour que `settings` lise `backend/.env` et que le chemin de la
base Chroma (`./data/chroma`) corresponde exactement à celui de l'API. `truststore` règle les
erreurs SSL dues à l'inspection TLS Zscaler (cf. `CLAUDE.md`).

In [ ]:
# %pip install truststore
# %pip install tenacity
# %pip install chromadb
# %pip install rank_bm25
# %pip install mistralai

  Using cached eval_type_backport-0.4.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached jsonpath_python-1.1.6-py3-none-any.whl.metadata (17 kB)
  Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached zipp-4.1.0-py3-none-any.whl.metadata (3.6 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------- ----- 1.0/1.2 MB 7.8 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 5.3 MB/s  0:00:00
Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl (219 kB)
Using cached opentelemetry_api-1.39.1-py3-none-any.whl (66 kB)
Using cached importlib_metadata-8.7.1-py3-none-any.whl (27 kB)
Using cached eval_type_backport-0.4.0-py3-none-any.whl (6.4 kB)
Using cached jsonpath_python-1.1.6-py3-none-any.whl (14 kB)
Us

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.43.0 requires opentelemetry-api==1.43.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.43.0 requires opentelemetry-semantic-conventions==0.64b0, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.


In [14]:
import os
import sys
from pathlib import Path

# notebooks/ -> ../backend
BACKEND = (Path.cwd() / ".." / "backend").resolve()
if not (BACKEND / "app").exists():
    raise RuntimeError(f"Backend introuvable à {BACKEND} — lancez ce notebook depuis notebooks/")

os.chdir(BACKEND)            # pour que settings.data_dir == backend/data (même Chroma que l'API)
sys.path.insert(0, str(BACKEND))

import truststore
truststore.inject_into_ssl()  # SSL via le magasin Windows (Zscaler)

from app.config import settings
from app.services import hf_client, vector_store, mistral_client

n_chunks = vector_store._collection().count()
print(f"Collection Chroma : {n_chunks} chunks indexés")
print(f"LLM génération : {settings.MISTRAL_MODEL}")
print(f"Embeddings       : {settings.HF_EMBEDDING_MODEL}")
print(f"RAG_TOP_K        : {settings.RAG_TOP_K}")
if n_chunks == 0:
    print("\n⚠️  Aucun chunk indexé : ingestez des documents avant de lancer le benchmark.")

Collection Chroma : 178 chunks indexés
LLM génération : mistral-small-latest
Embeddings       : intfloat/multilingual-e5-large
RAG_TOP_K        : 5


## 2. Jeu d'évaluation

Chaque entrée = une `question` + une `reference` (réponse de référence écrite à la main).
La référence sert à `context_precision` et `context_recall`.

> Les questions ci-dessous ciblent les documents **actuellement indexés** (BERT, schémas
> électriques). **À adapter / étoffer** à votre corpus : un benchmark crédible vise ~20-50
> questions variées (factuelles, multi-passages, et quelques-unes *sans réponse* dans le corpus
> pour tester l'aveu d'ignorance).

In [15]:
eval_set = [
    {
        "question": "Que signifie l'acronyme BERT ?",
        "reference": "BERT signifie Bidirectional Encoder Representations from Transformers.",
    },
    {
        "question": "Sur quelles deux tâches non supervisées BERT est-il pré-entraîné ?",
        "reference": (
            "BERT est pré-entraîné sur le Masked Language Model (MLM), où des tokens aléatoires "
            "sont masqués puis prédits, et sur la Next Sentence Prediction (NSP)."
        ),
    },
    {
        "question": "Quel pourcentage de tokens est masqué lors du pré-entraînement MLM de BERT ?",
        "reference": "15 % des tokens WordPiece de chaque séquence sont masqués aléatoirement.",
    },
    {
        "question": "Quel token spécial est ajouté au début de chaque séquence d'entrée de BERT ?",
        "reference": (
            "Le token spécial [CLS] est ajouté au début de chaque séquence ; son état caché "
            "final sert de représentation agrégée pour les tâches de classification."
        ),
    },
    {
        "question": "À quoi sert un schéma de circuit électrique (circuit diagram) ?",
        "reference": (
            "Un schéma de circuit représente, à l'aide de symboles normalisés, les composants "
            "d'un circuit électrique et les connexions électriques entre eux."
        ),
    },
    {
        "question": "En électronique, que représente le symbole de la mise à la terre (ground) ?",
        "reference": (
            "Le symbole de mise à la terre (ground) représente un point de référence commun à "
            "potentiel zéro dans le circuit, souvent un retour commun ou une connexion à la terre."
        ),
    },
]
print(f"{len(eval_set)} questions dans le jeu d'évaluation")

6 questions dans le jeu d'évaluation


## 3. Exécution du pipeline RAG sur chaque question

`run_rag` reproduit le cœur de `post_chat` (sans l'authentification ni la persistance des
conversations). Pour chaque question on récupère la **réponse générée** et la **liste des
passages récupérés** (les `retrieved_contexts` que RAGAS analysera).

In [16]:
def run_rag(question: str, top_k: int | None = None, document_ids: list[str] | None = None):
    """Exécute le pipeline RAG complet et renvoie (réponse, contextes, chunks bruts)."""
    top_k = top_k or settings.RAG_TOP_K
    embedding = hf_client.embed_texts([question])[0]
    chunks = vector_store.hybrid_query(embedding, question, top_k=top_k, document_ids=document_ids)
    answer = mistral_client.generate_answer(question, chunks)
    contexts = [c.text for c in chunks]
    return answer, contexts, chunks

In [17]:
samples = []
for i, item in enumerate(eval_set, 1):
    answer, contexts, chunks = run_rag(item["question"])
    samples.append(
        {
            "user_input": item["question"],
            "retrieved_contexts": contexts,
            "response": answer,
            "reference": item["reference"],
        }
    )
    srcs = ", ".join(sorted({f"{c.document_name} p.{c.page}" for c in chunks}))
    print(f"[{i}/{len(eval_set)}] {item['question']}")
    print(f"    → {answer[:160]}{'…' if len(answer) > 160 else ''}")
    print(f"    sources: {srcs}\n")

HfHubHTTPError: 403 Forbidden: None.
Cannot access content at: https://huggingface.co/api/models/intfloat/multilingual-e5-large.
Make sure your token has the correct permissions.

## 4. Configuration du juge RAGAS (Mistral + embeddings du projet)

- **LLM juge** : `ChatMistralAI` (même API que la génération). On peut prendre un modèle plus
  fort que celui de production pour juger (ex. `mistral-large-latest`).
- **Embeddings** : on enveloppe `hf_client.embed_texts` dans une classe `langchain` minimale,
  pour que `response_relevancy` utilise **exactement** les embeddings du système.

In [ ]:
from langchain_core.embeddings import Embeddings
from langchain_mistralai import ChatMistralAI

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


class ProjectEmbeddings(Embeddings):
    """Adapte les embeddings HF du projet à l'interface langchain pour RAGAS."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return hf_client.embed_texts(list(texts))

    def embed_query(self, text: str) -> list[float]:
        return hf_client.embed_texts([text])[0]


# mistral-large-latest = meilleur juge ; bascule sur le modèle de prod s'il n'est pas dispo.
JUDGE_MODEL = "mistral-large-latest"

judge_llm = LangchainLLMWrapper(
    ChatMistralAI(model=JUDGE_MODEL, api_key=settings.MISTRAL_API_KEY, temperature=0)
)
judge_embeddings = LangchainEmbeddingsWrapper(ProjectEmbeddings())
print(f"Juge : {JUDGE_MODEL}")

## 5. Évaluation RAGAS

In [ ]:
from ragas import EvaluationDataset, evaluate
from ragas.run_config import RunConfig
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

dataset = EvaluationDataset.from_list(samples)

metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
]

# max_workers bas : l'API Mistral (free/dev tier) limite le débit ; évite les 429.
result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=RunConfig(max_workers=4),
)
result

## 6. Résultats détaillés et sauvegarde

Scores par question (1.0 = parfait) + moyennes. Le CSV horodaté permet de comparer les runs
(avant/après un changement de `RAG_TOP_K`, de modèle d'embeddings, de prompt, etc.).

In [ ]:
import pandas as pd

df = result.to_pandas()
metric_cols = [c for c in df.columns if c in {
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
}]

print("=== Scores moyens ===")
print(df[metric_cols].mean().round(3).to_string())

with pd.option_context("display.max_colwidth", 80):
    display(df[["user_input", *metric_cols]].round(3))

In [ ]:
from datetime import datetime

out_dir = BACKEND.parent / "notebooks" / "benchmark_results"
out_dir.mkdir(exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = out_dir / f"ragas_{stamp}.csv"
df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Résultats enregistrés : {out_path}")

## 7. Lecture des résultats

- **Faithfulness bas, recall haut** : le retrieval ramène la bonne info mais le LLM hallucine
  ou extrapole → durcir le `SYSTEM_PROMPT`, baisser la température.
- **Context recall bas** : le retrieval rate les bons passages → jouer sur `RAG_TOP_K`, le
  chunking, ou la pondération dense/BM25 (RRF).
- **Context precision bas** : trop de passages hors-sujet remontés → baisser `top_k` ou
  ajouter un re-ranking.
- **Response relevancy bas** : la réponse part à côté de la question même avec un bon contexte
  → prompt de génération.

**Limites** : (1) le juge est lui-même un LLM (Mistral) — les scores sont indicatifs, pas
absolus ; (2) ces métriques portent sur le **texte** ; l'évaluation de la pertinence des
**figures** retournées nécessite un protocole dédié (hors périmètre RAGAS standard) ; (3)
étoffer le jeu d'évaluation est le levier n° 1 pour des chiffres fiables.